# Time Series Momentum (TSMOM)
## A Complete Guide: Theory, Math, and Implementation

**Author:** Educational Resource for Quantitative Finance  
**Based on:** Moskowitz, Ooi, and Pedersen (2012)

---

### What You'll Learn
1. What TSMOM is and why it works
2. The mathematical foundations
3. How to implement it from scratch
4. Real examples with code
5. Resources for further learning

---
# Part 1: Introduction

## 1.1 What is Time Series Momentum?

**Time Series Momentum (TSMOM)** is a trading strategy based on a simple observation:

> *"Assets that have gone up tend to keep going up. Assets that have gone down tend to keep going down."*

This is different from traditional "momentum" in finance:

| | Cross-Sectional Momentum | Time Series Momentum |
|---|---|---|
| **Comparison** | Asset vs. other assets | Asset vs. itself |
| **Signal** | Outperformance vs. peers | Positive/negative past return |
| **Position** | Long winners, short losers | Long if up, short if down |
| **Example** | "Apple beat Google, buy Apple" | "Apple is up 10%, buy Apple" |

## 1.2 Why Does It Work?

Several theories explain why momentum exists:

1. **Behavioral Biases**
   - Investors underreact to new information initially
   - Then overreact as the trend becomes obvious
   - This creates predictable price patterns

2. **Slow Information Diffusion**
   - Not everyone gets information at the same time
   - Prices adjust gradually, not instantly

3. **Herding Behavior**
   - Investors follow trends and each other
   - Creates self-reinforcing price movements

4. **Risk Premium**
   - Momentum strategies crash occasionally
   - Returns may be compensation for this crash risk

---
# Part 2: The Math

## 2.1 Basic Notation

Let's define our variables:

| Symbol | Meaning |
|--------|--------|
| $P_t$ | Price at time $t$ |
| $r_t$ | Return at time $t$ |
| $k$ | Lookback period (e.g., 252 days = 1 year) |
| $\sigma_t$ | Volatility at time $t$ |

## 2.2 Step 1: Calculate Returns

### Simple Return
$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$

**Example:** If price goes from \$100 to \$105:
$$r_t = \frac{105 - 100}{100} = 0.05 = 5\%$$

### Log Return
$$r_t^{\log} = \ln\left(\frac{P_t}{P_{t-1}}\right)$$

**Why log returns?** They're additive over time:
$$r_{t-k:t}^{\log} = \sum_{i=t-k+1}^{t} r_i^{\log}$$

## 2.3 Step 2: Calculate Lookback Return

The **lookback return** measures performance over the past $k$ periods:

$$r_{t-k:t} = \frac{P_t - P_{t-k}}{P_{t-k}}$$

**Example:** 12-month lookback (k=252 trading days)
- Price 1 year ago: \$100
- Price today: \$115
- Lookback return: $(115 - 100) / 100 = 15\%$

## 2.4 Step 3: Generate the Signal

The **TSMOM signal** is beautifully simple:

$$\boxed{\text{signal}_t = \text{sign}(r_{t-k:t})}$$

Where:
$$\text{sign}(x) = \begin{cases} +1 & \text{if } x > 0 \\ 0 & \text{if } x = 0 \\ -1 & \text{if } x < 0 \end{cases}$$

**Interpretation:**
- Signal = **+1**: Go **LONG** (buy the asset)
- Signal = **-1**: Go **SHORT** (sell/short the asset)

### Decision Rule

```
IF past k-period return > 0:
    BUY (go long)
ELSE:
    SELL (go short)
```

## 2.5 Step 4: Calculate Strategy Return

The **strategy return** for one period is:

$$\boxed{r_t^{\text{TSMOM}} = \text{signal}_{t-1} \times r_t}$$

**Why $\text{signal}_{t-1}$?** Because we use yesterday's signal to trade today.

### How It Works

| Signal | Actual Return | Strategy Return | Outcome |
|--------|---------------|-----------------|--------|
| +1 (Long) | +5% | +1 × 5% = **+5%** | Correct |
| +1 (Long) | -3% | +1 × -3% = **-3%** | Wrong |
| -1 (Short) | -4% | -1 × -4% = **+4%** | Correct |
| -1 (Short) | +2% | -1 × 2% = **-2%** | Wrong |

**Key insight:** When shorting, you profit when prices fall!

## 2.6 Step 5: Cumulative Returns

To track total performance over time:

### Compound Returns
$$\text{Cumulative Return}_T = \prod_{t=1}^{T} (1 + r_t^{\text{TSMOM}}) - 1$$

**Example:** Three days of returns: +2%, -1%, +3%
$$\text{Cumulative} = (1.02)(0.99)(1.03) - 1 = 1.0403 - 1 = 4.03\%$$

### Portfolio Value
Starting with \$100:
$$V_T = V_0 \times \prod_{t=1}^{T} (1 + r_t^{\text{TSMOM}})$$

## 2.7 Advanced: Volatility Scaling

Professional implementations **scale positions by volatility** so that:
- Risky assets get smaller positions
- Stable assets get larger positions
- All positions contribute similar risk

### Realized Volatility

$$\sigma_t = \sqrt{\frac{1}{n-1} \sum_{i=1}^{n} (r_{t-i} - \bar{r})^2}$$

**Annualized:** $\sigma_{\text{annual}} = \sigma_{\text{daily}} \times \sqrt{252}$

### Scaled Position Size

$$w_t = \text{signal}_t \times \frac{\sigma_{\text{target}}}{\sigma_t}$$

Where $\sigma_{\text{target}}$ is your desired volatility (e.g., 10% annually).

**Example:**
- Target volatility: 10%
- Asset volatility: 20%
- Position size: $10\% / 20\% = 0.5$ (half position)

---
# Part 3: Implementation

Let's implement TSMOM from scratch in Python.

In [ ]:
# Required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# For fetching real data (optional)
# pip install yfinance
# import yfinance as yf

print("Libraries loaded!")

In [ ]:
def calculate_lookback_return(prices, lookback):
    """
    Calculate the return over the past 'lookback' periods.
    
    Formula: r = (P_t - P_{t-k}) / P_{t-k}
    """
    return (prices - prices.shift(lookback)) / prices.shift(lookback)


def generate_signal(lookback_returns):
    """
    Generate TSMOM signal: +1 if positive, -1 if negative.
    
    Formula: signal = sign(r_{t-k:t})
    """
    return np.sign(lookback_returns)


def calculate_strategy_returns(signals, daily_returns):
    """
    Calculate strategy returns.
    
    Formula: r_tsmom = signal_{t-1} * r_t
    Note: We use shifted signal (yesterday's signal for today's trade)
    """
    return signals.shift(1) * daily_returns


def tsmom_backtest(prices, lookback=252):
    """
    Complete TSMOM backtest.
    
    Parameters:
    -----------
    prices : pd.Series
        Price series with datetime index
    lookback : int
        Lookback period in days (default: 252 = 1 year)
    
    Returns:
    --------
    pd.DataFrame with all calculations
    """
    df = pd.DataFrame(index=prices.index)
    df['price'] = prices
    
    # Step 1: Daily returns
    df['daily_return'] = prices.pct_change()
    
    # Step 2: Lookback return
    df['lookback_return'] = calculate_lookback_return(prices, lookback)
    
    # Step 3: Signal
    df['signal'] = generate_signal(df['lookback_return'])
    
    # Step 4: Strategy return
    df['tsmom_return'] = calculate_strategy_returns(df['signal'], df['daily_return'])
    
    # Step 5: Cumulative returns
    df['cumulative_tsmom'] = (1 + df['tsmom_return'].fillna(0)).cumprod()
    df['cumulative_buyhold'] = (1 + df['daily_return'].fillna(0)).cumprod()
    
    return df

## 3.1 Example with Synthetic Data

In [ ]:
# Generate synthetic price data with trends
np.random.seed(42)

# Create 3 years of daily data
n_days = 252 * 3
dates = pd.date_range('2021-01-01', periods=n_days, freq='B')  # Business days

# Create trending price series
returns = []
trend = 0.0005  # Start with uptrend

for i in range(n_days):
    # Change trend direction periodically
    if i % 60 == 0:  # Every ~3 months
        trend = np.random.choice([-0.0008, -0.0004, 0.0004, 0.0008])
    
    daily_return = trend + np.random.normal(0, 0.015)
    returns.append(daily_return)

# Convert to prices
prices = 100 * np.cumprod(1 + np.array(returns))
price_series = pd.Series(prices, index=dates, name='Synthetic Asset')

# Plot the price
plt.figure(figsize=(12, 4))
plt.plot(price_series)
plt.title('Synthetic Price Series with Trending Regimes')
plt.ylabel('Price')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Run TSMOM backtest
results = tsmom_backtest(price_series, lookback=63)  # 3-month lookback

# Show sample of results
print("Sample of backtest results:")
print(results[['price', 'lookback_return', 'signal', 'tsmom_return']].dropna().head(10))

In [ ]:
# Plot cumulative returns comparison
plt.figure(figsize=(12, 6))

plt.plot(results['cumulative_tsmom'], label='TSMOM Strategy', linewidth=2)
plt.plot(results['cumulative_buyhold'], label='Buy & Hold', linewidth=2, alpha=0.7)

plt.title('TSMOM vs Buy & Hold Performance')
plt.ylabel('Portfolio Value (Starting = $1)')
plt.xlabel('Date')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Print final returns
print(f"\nFinal Results:")
print(f"TSMOM Total Return: {(results['cumulative_tsmom'].iloc[-1] - 1) * 100:.2f}%")
print(f"Buy & Hold Return:  {(results['cumulative_buyhold'].iloc[-1] - 1) * 100:.2f}%")

In [ ]:
# Visualize signals on price chart
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Price with signal colors
ax1 = axes[0]
ax1.plot(results['price'], color='black', linewidth=1)

# Color background based on signal
for i in range(1, len(results)):
    if results['signal'].iloc[i] == 1:
        ax1.axvspan(results.index[i-1], results.index[i], alpha=0.1, color='green')
    elif results['signal'].iloc[i] == -1:
        ax1.axvspan(results.index[i-1], results.index[i], alpha=0.1, color='red')

ax1.set_ylabel('Price')
ax1.set_title('Price with TSMOM Signals (Green = Long, Red = Short)')
ax1.grid(True, alpha=0.3)

# Lookback return
ax2 = axes[1]
colors = ['green' if x > 0 else 'red' for x in results['lookback_return'].fillna(0)]
ax2.bar(results.index, results['lookback_return'].fillna(0), color=colors, alpha=0.7, width=1)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.set_ylabel('Lookback Return')
ax2.set_title('Lookback Return (Determines Signal)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3.2 Example with Real Data

In [ ]:
# Uncomment to use real data (requires yfinance: pip install yfinance)

# import yfinance as yf

# # Download S&P 500 ETF data
# spy = yf.download('SPY', start='2010-01-01', end='2024-01-01')['Adj Close']

# # Run backtest
# real_results = tsmom_backtest(spy, lookback=252)

# # Plot
# plt.figure(figsize=(12, 6))
# plt.plot(real_results['cumulative_tsmom'], label='TSMOM')
# plt.plot(real_results['cumulative_buyhold'], label='Buy & Hold')
# plt.title('TSMOM on S&P 500 (SPY)')
# plt.legend()
# plt.show()

print("Uncomment the code above to test with real market data!")

---
# Part 4: Performance Metrics

How do we evaluate if a strategy is good?

In [ ]:
def calculate_metrics(returns, risk_free_rate=0.02):
    """
    Calculate comprehensive performance metrics.
    
    Parameters:
    -----------
    returns : pd.Series
        Daily returns
    risk_free_rate : float
        Annual risk-free rate (default: 2%)
    """
    returns = returns.dropna()
    
    # Basic stats
    total_return = (1 + returns).prod() - 1
    n_years = len(returns) / 252
    annual_return = (1 + total_return) ** (1/n_years) - 1
    annual_vol = returns.std() * np.sqrt(252)
    
    # Sharpe Ratio
    sharpe = (annual_return - risk_free_rate) / annual_vol if annual_vol > 0 else 0
    
    # Win Rate
    win_rate = (returns > 0).sum() / len(returns)
    
    # Maximum Drawdown
    cumulative = (1 + returns).cumprod()
    peak = cumulative.expanding().max()
    drawdown = (cumulative - peak) / peak
    max_drawdown = drawdown.min()
    
    # Calmar Ratio (return / max drawdown)
    calmar = annual_return / abs(max_drawdown) if max_drawdown != 0 else 0
    
    return {
        'Total Return': f"{total_return*100:.2f}%",
        'Annual Return': f"{annual_return*100:.2f}%",
        'Annual Volatility': f"{annual_vol*100:.2f}%",
        'Sharpe Ratio': f"{sharpe:.2f}",
        'Win Rate': f"{win_rate*100:.2f}%",
        'Max Drawdown': f"{max_drawdown*100:.2f}%",
        'Calmar Ratio': f"{calmar:.2f}"
    }


# Calculate metrics for our backtest
tsmom_metrics = calculate_metrics(results['tsmom_return'])
buyhold_metrics = calculate_metrics(results['daily_return'])

print("=" * 50)
print("PERFORMANCE COMPARISON")
print("=" * 50)
print(f"{'Metric':<20} {'TSMOM':<15} {'Buy & Hold':<15}")
print("-" * 50)
for key in tsmom_metrics:
    print(f"{key:<20} {tsmom_metrics[key]:<15} {buyhold_metrics[key]:<15}")

## 4.1 Understanding the Metrics

### Sharpe Ratio
$$\text{Sharpe} = \frac{R_p - R_f}{\sigma_p}$$

- Measures **risk-adjusted return**
- Higher is better
- < 1: Below average | 1-2: Good | > 2: Excellent

### Maximum Drawdown
$$\text{MDD} = \max_{t} \left( \frac{\text{Peak}_t - \text{Value}_t}{\text{Peak}_t} \right)$$

- Largest peak-to-trough decline
- Lower (closer to 0) is better
- -20% means you lost 20% from the highest point

### Win Rate
$$\text{Win Rate} = \frac{\text{Profitable Days}}{\text{Total Days}}$$

- Percentage of positive return days
- > 50% means more winning days than losing days
- Note: You can have < 50% win rate but still be profitable if wins are bigger than losses!

---
# Part 5: Variations and Improvements

## 5.1 Different Lookback Periods

In [ ]:
# Test different lookback periods
lookbacks = [21, 63, 126, 252]  # 1, 3, 6, 12 months
lookback_names = ['1 Month', '3 Months', '6 Months', '12 Months']

plt.figure(figsize=(12, 6))

for lb, name in zip(lookbacks, lookback_names):
    result = tsmom_backtest(price_series, lookback=lb)
    plt.plot(result['cumulative_tsmom'], label=f'TSMOM ({name})', linewidth=1.5)

plt.plot(results['cumulative_buyhold'], label='Buy & Hold', linewidth=2, color='black', linestyle='--')

plt.title('TSMOM Performance with Different Lookback Periods')
plt.ylabel('Portfolio Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5.2 Volatility-Scaled TSMOM

In [ ]:
def tsmom_vol_scaled(prices, lookback=252, vol_lookback=60, target_vol=0.10):
    """
    TSMOM with volatility scaling.
    
    Parameters:
    -----------
    prices : pd.Series
        Price series
    lookback : int
        Signal lookback period
    vol_lookback : int
        Volatility calculation period
    target_vol : float
        Target annualized volatility
    """
    df = pd.DataFrame(index=prices.index)
    df['price'] = prices
    df['daily_return'] = prices.pct_change()
    
    # Lookback return and signal
    df['lookback_return'] = calculate_lookback_return(prices, lookback)
    df['signal'] = generate_signal(df['lookback_return'])
    
    # Rolling volatility (annualized)
    df['volatility'] = df['daily_return'].rolling(vol_lookback).std() * np.sqrt(252)
    
    # Position size: scale by volatility
    df['position_size'] = target_vol / df['volatility']
    df['position_size'] = df['position_size'].clip(upper=2)  # Cap at 2x leverage
    
    # Scaled signal
    df['scaled_signal'] = df['signal'] * df['position_size']
    
    # Strategy returns
    df['tsmom_return'] = df['scaled_signal'].shift(1) * df['daily_return']
    df['cumulative_tsmom'] = (1 + df['tsmom_return'].fillna(0)).cumprod()
    
    return df


# Compare scaled vs unscaled
unscaled = tsmom_backtest(price_series, lookback=63)
scaled = tsmom_vol_scaled(price_series, lookback=63, target_vol=0.15)

plt.figure(figsize=(12, 6))
plt.plot(unscaled['cumulative_tsmom'], label='TSMOM (Unscaled)')
plt.plot(scaled['cumulative_tsmom'], label='TSMOM (Vol Scaled)')
plt.plot(unscaled['cumulative_buyhold'], label='Buy & Hold', linestyle='--')
plt.title('TSMOM: Unscaled vs Volatility-Scaled')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
# Part 6: Common Pitfalls

## Things to Watch Out For

### 1. Look-Ahead Bias
**Problem:** Using future information to make decisions  
**Solution:** Always use `signal.shift(1)` - yesterday's signal for today's trade

### 2. Survivorship Bias
**Problem:** Only testing on assets that still exist today  
**Solution:** Include delisted assets in your backtest

### 3. Transaction Costs
**Problem:** Ignoring trading costs  
**Solution:** Subtract realistic costs (0.1-0.5% per trade)

### 4. Overfitting
**Problem:** Optimizing parameters to fit historical data perfectly  
**Solution:** Use out-of-sample testing, keep it simple

### 5. Momentum Crashes
**Problem:** Momentum strategies can crash hard during market reversals  
**Solution:** Use volatility scaling, diversify across assets

In [ ]:
# Example: Adding transaction costs

def tsmom_with_costs(prices, lookback=252, cost_per_trade=0.001):
    """
    TSMOM with transaction costs.
    
    cost_per_trade: Cost as fraction of trade value (0.001 = 0.1%)
    """
    df = tsmom_backtest(prices, lookback)
    
    # Detect signal changes (trades)
    df['signal_change'] = df['signal'].diff().abs() / 2  # 0, 0.5, or 1
    df['signal_change'] = df['signal_change'].fillna(0)
    
    # Subtract transaction costs
    df['tsmom_return_net'] = df['tsmom_return'] - (df['signal_change'] * cost_per_trade)
    df['cumulative_tsmom_net'] = (1 + df['tsmom_return_net'].fillna(0)).cumprod()
    
    return df


# Compare with and without costs
no_costs = tsmom_backtest(price_series, lookback=63)
with_costs = tsmom_with_costs(price_series, lookback=63, cost_per_trade=0.002)  # 0.2% per trade

plt.figure(figsize=(12, 5))
plt.plot(no_costs['cumulative_tsmom'], label='TSMOM (No Costs)')
plt.plot(with_costs['cumulative_tsmom_net'], label='TSMOM (0.2% Costs)')
plt.title('Impact of Transaction Costs')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

n_trades = with_costs['signal_change'].sum()
print(f"Number of round-trip trades: {n_trades:.0f}")
print(f"Total transaction costs: {n_trades * 0.002 * 100:.2f}%")

---
# Part 7: Summary

## Key Takeaways

### The TSMOM Formula

$$\boxed{r_t^{\text{TSMOM}} = \text{sign}\left(\frac{P_{t-1} - P_{t-1-k}}{P_{t-1-k}}\right) \times \frac{P_t - P_{t-1}}{P_{t-1}}}$$

In words:
1. Look at past k-period return
2. If positive, go long; if negative, go short
3. Your return = signal × actual return

### Quick Reference

| Step | Formula | Code |
|------|---------|------|
| Lookback Return | $\frac{P_t - P_{t-k}}{P_{t-k}}$ | `(P - P.shift(k)) / P.shift(k)` |
| Signal | $\text{sign}(r_{lookback})$ | `np.sign(lookback_return)` |
| Strategy Return | $\text{signal}_{t-1} \times r_t$ | `signal.shift(1) * daily_return` |
| Cumulative | $\prod(1 + r)$ | `(1 + returns).cumprod()` |

---
# Part 8: Resources

## Academic Papers

### Essential Reading
1. **Moskowitz, Ooi, Pedersen (2012)** - "Time Series Momentum"  
   *Journal of Financial Economics*  
   The original paper that documented TSMOM across 58 futures markets.
   
2. **Asness, Moskowitz, Pedersen (2013)** - "Value and Momentum Everywhere"  
   *The Journal of Finance*  
   Shows momentum works across all asset classes globally.

3. **Hurst, Ooi, Pedersen (2017)** - "A Century of Evidence on Trend-Following Investing"  
   *Journal of Portfolio Management*  
   100 years of trend-following evidence.

### Advanced Reading
4. **Baltas, Kosowski (2013)** - "Momentum Strategies in Futures Markets"  
5. **Dudler, Gmuer, Malamud (2022)** - "Risk-Adjusted Time Series Momentum"  
6. **Huang, Li, Wang, Zhou (2020)** - "Time Series Momentum: Is it There?"  

## Books

1. **"Expected Returns"** by Antti Ilmanen  
   Comprehensive guide to factor investing including momentum.

2. **"Quantitative Momentum"** by Wesley Gray & Jack Vogel  
   Practical guide to momentum strategies.

3. **"Efficiently Inefficient"** by Lasse Pedersen  
   Academic perspective on trading strategies.

4. **"Following the Trend"** by Andreas Clenow  
   Practical trend-following implementation.

## Online Resources

### Data Sources
- **Yahoo Finance** (free): `yfinance` Python library
- **Quandl** (free/paid): Historical futures data
- **Alpha Vantage** (free): Stock data API
- **FRED** (free): Economic data

### Learning Platforms
- **Quantopian Lectures** (archived): Great intro to quant finance
- **QuantConnect**: Algorithmic trading platform with tutorials
- **Coursera**: "Machine Learning for Trading" by Georgia Tech

### Blogs & Websites
- **AQR Capital** (aqr.com): Research papers and insights
- **Alpha Architect** (alphaarchitect.com): Factor investing research
- **Quantpedia** (quantpedia.com): Strategy database
- **SSRN** (ssrn.com): Academic paper repository

## Python Libraries

```python
# Data
pip install yfinance pandas-datareader

# Analysis
pip install numpy pandas scipy statsmodels

# Visualization
pip install matplotlib seaborn plotly

# Backtesting
pip install backtrader bt vectorbt

# Risk/Performance
pip install empyrical pyfolio quantstats
```

---
# Part 9: Exercises

Test your understanding!

### Exercise 1: Basic Signal
Given prices [100, 105, 103, 108, 110] and lookback=2:
- Calculate the lookback return at each point
- Determine the signal at each point

### Exercise 2: Strategy Return
If your signal is +1 (long) and the asset drops 3%, what is your strategy return?

### Exercise 3: Implementation
Modify the code to:
1. Use a 6-month lookback
2. Add a minimum threshold (only trade if |lookback_return| > 5%)
3. Compare performance with and without the threshold

### Exercise 4: Multi-Asset
Download data for SPY, TLT, and GLD. Run TSMOM on each and create an equal-weighted portfolio.

### Exercise 5: Analysis
Why might TSMOM underperform during certain market conditions? Identify periods in the backtest where this happens.

---

## Congratulations!

You now understand:
- What TSMOM is and why it works
- The mathematical foundations
- How to implement it from scratch
- How to evaluate performance
- Common pitfalls to avoid

**Next Steps:**
1. Run the code with real market data
2. Experiment with different lookback periods
3. Add volatility scaling
4. Build a multi-asset portfolio
5. Read the original Moskowitz paper

---

*Happy Trading!*